# ACE-Net — Stage-1 Emotion Eval (Tables 2/3)  ·  PERSON B

Evaluates the trained Stage-1 emotion extractors on their held-out test splits
and prints **Accuracy + Weighted-F1 + confusion matrix** per dataset
(paper Tables 2 and 3). No training — fast.

Set Runtime → T4 GPU (or even CPU works, just slower).

**You need uploaded:** `emotion_vectors.zip` on Drive (CREMA genuine + MELD),
and all four Stage-1 checkpoints (visual/speech x crema/meld).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Clone repo

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git log --oneline -1

## Install dependencies

In [ ]:
!pip -q install torch torchvision torchaudio transformers librosa pillow
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## Get the vectors

Mount Drive and unzip **`emotion_vectors.zip`** (upload it to your Drive root first).
Zip contains `CREMA-D/GENUINE_LastHalf`, `CREMA-D/GENUINE_FirstHalf`, and `MELD/` (with train/dev/test) at the right nesting so they unzip to `data/CREMA-D/...` and `data/MELD/...`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
ZIP = '/content/drive/MyDrive/emotion_vectors.zip'   # adjust if needed
DST = '/content/Baseline_Training/data'
os.makedirs(DST, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DST)
print('unzipped under data/:')
for root,dirs,_ in os.walk(DST):
    depth = root[len(DST):].count(os.sep)
    if depth < 2: print(' ', root)

## Upload Stage-1 checkpoints

In [ ]:
import os
os.makedirs('checkpoints', exist_ok=True)
from google.colab import files
up = files.upload()
for name in up:
    os.replace(name, f'checkpoints/{name}')
print(os.listdir('checkpoints'))

## Table 3 — Facial emotion (FV-LiteNet)

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage1 --branch visual --dataset crema
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage1 --branch visual --dataset meld

## Table 2 — Speech-Text emotion (MDCNN + cross-attention)

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage1 --branch speech_text --dataset crema
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage1 --branch speech_text --dataset meld

## Notes

These numbers come from time-capped Stage-1 training (few epochs on a 6GB
local GPU), so accuracy is under-converged vs the paper. They are valid,
just not fully trained. For better numbers, retrain Stage-1 longer on the T4
(`python -m src.train_stage1 --branch ... --dataset ... --epochs 40`).